# Brain Tumor Segmentation — U-Net
## Google Colab Training Notebook
### PCCOE MDM Capstone 2025-26
**Run AFTER completing classification training**

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import files
import os

uploaded = files.upload()  # upload kaggle.json
os.makedirs('/root/.kaggle', exist_ok=True)
os.system('cp kaggle.json /root/.kaggle/')
os.system('chmod 600 /root/.kaggle/kaggle.json')
print("Kaggle credentials ready")

In [ ]:
os.system('kaggle datasets download -d mateuszbuda/lgg-mri-segmentation -p /content/data/')
os.system('unzip -q /content/data/lgg-mri-segmentation.zip -d /content/data/LGG\ MRI\ Segmentation')
print("LGG dataset downloaded")

# Verify
import glob
masks = glob.glob('/content/data/LGG MRI Segmentation/kaggle_3m/**/*_mask.tif', recursive=True)
images = [m.replace('_mask.tif', '.tif') for m in masks]
valid = [(img, msk) for img, msk in zip(images, masks) if os.path.exists(img)]
print(f"Valid image-mask pairs found: {len(valid)}")

In [ ]:
!pip install -q tensorflow==2.17.0 opencv-python-headless==4.10.0.84 albumentations==1.4.11 scikit-learn==1.5.1 matplotlib==3.9.2 tqdm==4.66.4
print("Dependencies installed")

In [ ]:
!git clone https://github.com/mitali2744/Brain-tumor-diagnostic-system.git /content/project
import sys
sys.path.insert(0, '/content/project')

import config
config.SEGMENTATION_DATA_DIR = '/content/data/LGG MRI Segmentation/kaggle_3m'
config.MODELS_DIR = '/content/models'
config.SEGMENTATION_MODEL_PATH = '/content/models/unet_segmentation.h5'
os.makedirs(config.MODELS_DIR, exist_ok=True)
print("Config patched for Colab")

## EDA — Segmentation Dataset

In [ ]:
import numpy as np
import cv2
import glob
import matplotlib.pyplot as plt

data_dir = '/content/data/LGG MRI Segmentation/kaggle_3m'
masks = sorted(glob.glob(os.path.join(data_dir, '**', '*_mask.tif'), recursive=True))
images = [m.replace('_mask.tif', '.tif') for m in masks]
valid_pairs = [(img, msk) for img, msk in zip(images, masks) if os.path.exists(img)]

print(f"Total valid pairs: {len(valid_pairs)}")

# Compute tumor coverage per image
coverages = []
for img_path, msk_path in valid_pairs[:500]:  # sample 500 for speed
    mask = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)
    if mask is not None:
        coverage = np.sum(mask > 127) / mask.size * 100
        coverages.append(coverage)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(coverages, bins=30, color='steelblue', edgecolor='white')
plt.title('Tumor Pixel Coverage Distribution')
plt.xlabel('Tumor Coverage (%)')
plt.ylabel('Count')
plt.axvline(np.mean(coverages), color='red', linestyle='--',
            label=f'Mean={np.mean(coverages):.1f}%')
plt.legend()

plt.subplot(1, 2, 2)
has_tumor = sum(1 for c in coverages if c > 0)
no_tumor = len(coverages) - has_tumor
plt.bar(['Has Tumor', 'No Tumor'], [has_tumor, no_tumor],
        color=['#e74c3c', '#27ae60'])
plt.title('Images With vs Without Tumor')
plt.ylabel('Count')
for i, v in enumerate([has_tumor, no_tumor]):
    plt.text(i, v + 2, str(v), ha='center')

plt.tight_layout()
plt.savefig('/content/segmentation_eda.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
sample_pairs = [p for p in valid_pairs if cv2.imread(p[1], 0).max() > 0][:6]

for i, (img_path, msk_path) in enumerate(sample_pairs):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    msk = cv2.imread(msk_path, cv2.IMREAD_GRAYSCALE)

    row = i // 2
    col_offset = (i % 2) * 2

    axes[row][col_offset].imshow(img)
    axes[row][col_offset].set_title(f'MRI Slice {i+1}', fontsize=9)
    axes[row][col_offset].axis('off')

    axes[row][col_offset+1].imshow(msk, cmap='Reds')
    axes[row][col_offset+1].set_title(f'Tumor Mask {i+1}', fontsize=9)
    axes[row][col_offset+1].axis('off')

plt.suptitle('Sample MRI Slices with Ground Truth Masks', fontsize=13)
plt.tight_layout()
plt.savefig('/content/sample_masks.png', dpi=150)
plt.show()

## Train U-Net

In [ ]:
from src.segmentation.unet import get_compiled_unet
from src.segmentation.dataset import build_segmentation_generators

print("Loading segmentation dataset...")
train_gen, val_gen = build_segmentation_generators()
print(f"Train batches: {len(train_gen)} | Val batches: {len(val_gen)}")

print("Building U-Net...")
model = get_compiled_unet()
model.summary()

# Callbacks
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    '/content/models/unet_best.h5',
    monitor='val_dice_coefficient',
    save_best_only=True,
    mode='max',
    verbose=1
)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_dice_coefficient',
    patience=8,
    mode='max',
    restore_best_weights=True,
    verbose=1
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_dice_coefficient',
    factor=0.5,
    patience=4,
    mode='max',
    min_lr=1e-7,
    verbose=1
)

print("\nTraining U-Net...")
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=50,
    callbacks=[checkpoint, early_stop, reduce_lr],
    verbose=1
)
print("Training complete")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['dice_coefficient'], label='Train Dice', color='steelblue')
axes[0].plot(history.history['val_dice_coefficient'], label='Val Dice', color='tomato')
axes[0].set_title('Dice Coefficient'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history.history['loss'], label='Train Loss', color='steelblue')
axes[1].plot(history.history['val_loss'], label='Val Loss', color='tomato')
axes[1].set_title('Combined Loss (BCE + Dice)'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('U-Net Training History')
plt.tight_layout()
plt.savefig('/content/unet_history.png', dpi=150)
plt.show()

In [ ]:
print("\n--- Validation Set Evaluation ---")
results = model.evaluate(val_gen, verbose=1)
for name, val in zip(model.metrics_names, results):
    print(f"  {name}: {val:.4f}")

# Visual prediction samples
import numpy as np
sample_batch = val_gen[0]
images_batch, masks_batch = sample_batch
preds = model.predict(images_batch[:4], verbose=0)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
for i in range(4):
    axes[i][0].imshow(images_batch[i])
    axes[i][0].set_title('Original MRI'); axes[i][0].axis('off')
    axes[i][1].imshow(masks_batch[i].squeeze(), cmap='Reds')
    axes[i][1].set_title('Ground Truth'); axes[i][1].axis('off')
    pred_binary = (preds[i] > 0.5).astype(float)
    axes[i][2].imshow(pred_binary.squeeze(), cmap='Reds')
    axes[i][2].set_title('Predicted Mask'); axes[i][2].axis('off')
plt.suptitle('U-Net Predictions vs Ground Truth', fontsize=13)
plt.tight_layout()
plt.savefig('/content/unet_predictions.png', dpi=150)
plt.show()

In [ ]:
model.save('/content/models/unet_segmentation.h5')
print("U-Net model saved")

In [ ]:
from google.colab import files

files.download('/content/models/unet_segmentation.h5')
files.download('/content/unet_history.png')
files.download('/content/unet_predictions.png')
files.download('/content/segmentation_eda.png')
files.download('/content/sample_masks.png')
print("All files downloaded")

In [ ]:
import shutil
drive_path = '/content/drive/MyDrive/BrainTumorProject/models/'
os.makedirs(drive_path, exist_ok=True)
shutil.copy('/content/models/unet_segmentation.h5', drive_path)
shutil.copy('/content/unet_history.png', drive_path)
shutil.copy('/content/unet_predictions.png', drive_path)
print(f"Saved to Drive: {drive_path}")